In [1]:
import importlib
import _5_ServingColumns
importlib.reload(_5_ServingColumns)

<module '_5_ServingColumns' from '/home/nakyung/projects/BDAIFin/5MODEL/STAGE2/_5_ServingColumns.py'>

In [2]:
pwd

'/home/nakyung/projects/BDAIFin/5MODEL/STAGE2'

In [3]:
from pathlib import Path

base = Path("../../5DATA/dataset")
print("abs path:", base.resolve())
print("exists:", base.exists())
print("contents:")
for p in base.iterdir():
    print(" -", p.name)

abs path: /home/nakyung/projects/BDAIFin/5DATA/dataset
exists: True
contents:
 - train_stage2
 - TRAIN_stage1
 - TEST_stage2
 - test_stage1
 - TRAIN_stage2
 - CHECK_stage2
 - train_stage1
 - test.ipynb
 - test_stage2
 - check_stage1
 - TEST_stage1
 - check_stage2


In [4]:
from pathlib import Path
import pandas as pd

# 현재 노트북 위치 기준으로 "BDAIFin" 루트 자동 탐색(안전)
HERE = Path.cwd()
# /home/nakyung/projects/BDAIFin/5MODEL/STAGE2 같은 경로라면
ROOT = HERE
while ROOT.name != "BDAIFin" and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if ROOT.name != "BDAIFin":
    raise RuntimeError(f"Cannot find BDAIFin root from cwd={HERE}")

DATA_ROOT = ROOT / "5DATA"

TRAIN_IN  = DATA_ROOT / "dataset" / "train_stage2"
TEST_IN   = DATA_ROOT / "dataset" / "test_stage2"
ART_PATH  = DATA_ROOT / "artifacts" / "stage2_artifacts.json"
MCC_STATS_PATH = DATA_ROOT / "artifacts" / "mcc_stats.parquet"

TRAIN_OUT = DATA_ROOT / "dataset" / "TRAIN_STAGE2"
TEST_OUT  = DATA_ROOT / "dataset" / "TEST_STAGE2"

print("ROOT:", ROOT)
print("TRAIN_IN exists:", TRAIN_IN.exists(), "is_dir:", TRAIN_IN.is_dir())
print("TEST_IN  exists:", TEST_IN.exists(),  "is_dir:", TEST_IN.is_dir())

ROOT: /home/nakyung/projects/BDAIFin
TRAIN_IN exists: True is_dir: False
TEST_IN  exists: True is_dir: False


In [5]:
from _5_ServingColumns import fit_artifacts, build_features_101, save_json, sanity_report
print("Imported OK:", fit_artifacts, build_features_101)

Imported OK: <function fit_artifacts at 0x7f9f3f169000> <function build_features_101 at 0x7f9f3f168af0>


In [6]:
from pathlib import Path
import pandas as pd
import shutil

ART_PATH  = "../../5DATA/artifacts/stage2_artifacts.json"
MCC_STATS_PATH = "../../5DATA/artifacts/mcc_stats.parquet"


# read (폴더 parquet dataset)
df_train_raw = pd.read_parquet(TRAIN_IN)
df_test_raw  = pd.read_parquet(TEST_IN)

print("train raw:", df_train_raw.shape)
print("test  raw :", df_test_raw.shape)

# artifacts (train only)
artifacts, mcc_stats_df = fit_artifacts(df_train_raw)
save_json(artifacts, ART_PATH)
Path(MCC_STATS_PATH).parent.mkdir(parents=True, exist_ok=True)
mcc_stats_df.to_parquet(MCC_STATS_PATH, index=False)

print("saved artifacts:", ART_PATH)
print("saved mcc_stats:", MCC_STATS_PATH)

# build features
df_train_feat = build_features_101(df_train_raw, artifacts, mcc_stats_df)
df_test_feat  = build_features_101(df_test_raw,  artifacts, mcc_stats_df)

print("train feat:", df_train_feat.shape)
print("test  feat :", df_test_feat.shape)

sanity_report(df_train_feat)
sanity_report(df_test_feat)

# overwrite-safe save
for out in [TRAIN_OUT, TEST_OUT]:
    p = Path(out)
    if p.exists():
        shutil.rmtree(p)

df_train_feat.to_parquet(TRAIN_OUT, index=False)
df_test_feat.to_parquet(TEST_OUT, index=False)

print("saved TRAIN_OUT:", TRAIN_OUT)
print("saved TEST_OUT :", TEST_OUT)

train raw: (608430, 55)
test  raw : (113943, 55)
saved artifacts: ../../5DATA/artifacts/stage2_artifacts.json
saved mcc_stats: ../../5DATA/artifacts/mcc_stats.parquet
train feat: (608430, 65)
test  feat : (113943, 65)
=== Sanity Check ===
n_features: 63
missing: []
extra: []
=== Sanity Check ===
n_features: 63
missing: []
extra: []
saved TRAIN_OUT: /home/nakyung/projects/BDAIFin/5DATA/dataset/TRAIN_STAGE2
saved TEST_OUT : /home/nakyung/projects/BDAIFin/5DATA/dataset/TEST_STAGE2
